In [27]:
%load_ext autoreload
%autoreload 2



from pathlib import Path
from src.kb_engine.rag_engine import RagEngine
from src.kb_engine.ingest import IngestionHandler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import tempfile
persist_dir = Path("data/rag_engine_test")
rag_engine = RagEngine(
            name="anchor_rag3_test",
            persist_dir=persist_dir,
            ingestion_handler=IngestionHandler(),
            embedding_model="text-embedding-3-large",
        )
rag_engine.ingest(files=[Path("data/uploads/Flux_Capacitor_Technical_Reference_Manual.pdf")])



2026-03-04 14:56:52,028 - WARNING - No existing llama_index.core.graph_stores.simple found at data/rag_engine_test/text-embedding-3-large/index/default/graph_store.json. Initializing a new graph_store from scratch. 
2026-03-04 14:56:52,039 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-04 14:56:52,041 - INFO - Going to convert document batch...
2026-03-04 14:56:52,041 - INFO - Initializing pipeline for StandardPdfPipeline with options hash d125b8a2c18e6b7b87dbdce4ea6b7d04
2026-03-04 14:56:52,042 - INFO - Auto OCR model selected ocrmac.
2026-03-04 14:56:52,042 - INFO - Accelerator device: 'mps'


RagEngine initialized with and kwargs: {'name': 'anchor_rag3_test', 'persist_dir': PosixPath('data/rag_engine_test'), 'ingestion_handler': <src.kb_engine.ingest.IngestionHandler object at 0x3a08b88c0>, 'embedding_model': 'text-embedding-3-large'}


2026-03-04 14:56:52,879 - INFO - Accelerator device: 'mps'
2026-03-04 14:56:53,310 - INFO - Processing document Flux_Capacitor_Technical_Reference_Manual.pdf
2026-03-04 14:56:55,846 - INFO - Finished converting document Flux_Capacitor_Technical_Reference_Manual.pdf in 3.81 sec.


{"id_":"355e41a3-2fe6-4fa4-ad18-07352d0e1db5","embedding":null,"metadata":{"filepath":"data/uploads/Flux_Capacitor_Technical_Reference_Manual.pdf"},"excluded_embed_metadata_keys":[],"excluded_llm_metadata_keys":[],"relationships":{},"metadata_template":"{key}: {value}","metadata_separator":"\n","text_resource":{"embeddings":null,"text":"{\"schema_name\": \"DoclingDocument\", \"version\": \"1.9.0\", \"name\": \"Flux_Capacitor_Technical_Reference_Manual\", \"origin\": {\"mimetype\": \"application/pdf\", \"binary_hash\": 4142608075159877010, \"filename\": \"Flux_Capacitor_Technical_Reference_Manual.pdf\"}, \"furniture\": {\"self_ref\": \"#/furniture\", \"children\": [], \"content_layer\": \"furniture\", \"name\": \"_root_\", \"label\": \"unspecified\"}, \"body\": {\"self_ref\": \"#/body\", \"children\": [{\"$ref\": \"#/texts/0\"}, {\"$ref\": \"#/groups/0\"}, {\"$ref\": \"#/texts/5\"}, {\"$ref\": \"#/texts/6\"}, {\"$ref\": \"#/texts/7\"}, {\"$ref\": \"#/tables/0\"}, {\"$ref\": \"#/texts/8\

2026-03-04 14:56:57,000 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


12

In [30]:
result = rag_engine.query("what i the ttemper in the flux capacitor?")
from devtools import debug
debug(result)

2026-03-04 14:57:07,155 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-04 14:57:08,683 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


/var/folders/bf/4pk3n3917lv30ykyw5g_g0mw0000gn/T/ipykernel_46809/1504850663.py:3 <module>
    result: Response(
        response='The temperature in the flux capacitor is not explicitly mentioned in the provided context information.',
        source_nodes=[
            NodeWithScore(
                node=TextNode(
                    id_='54b13131-0325-4f4a-b880-2da08a684001',
                    embedding=None,
                    metadata={
                        'filepath': 'data/uploads/Flux_Capacitor_Technical_Reference_Manual.pdf',
                        'schema_name': 'docling_core.transforms.chunker.DocMeta',
                        'version': '1.0.0',
                        'doc_items': [
                            {
                                'self_ref': '#/texts/12',
                                'parent': {
                                    '$ref': '#/body',
                                },
                                'children': [],
                     

Response(response='The temperature in the flux capacitor is not explicitly mentioned in the provided context information.', source_nodes=[NodeWithScore(node=TextNode(id_='54b13131-0325-4f4a-b880-2da08a684001', embedding=None, metadata={'filepath': 'data/uploads/Flux_Capacitor_Technical_Reference_Manual.pdf', 'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/12', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 4, 'bbox': {'l': 78.0, 't': 527.0699705078125, 'r': 553.0, 'b': 493.81997050781246, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 203]}]}], 'headings': ['3. Mechanical & Thermal Properties'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 4142608075159877010, 'filename': 'Flux_Capacitor_Technical_Reference_Manual.pdf'}}, excluded_embed_metadata_keys=['schema_name', 'version', 'doc_items', 'origin'], excluded_llm_metadata_keys=['schema_name', 'vers

In [ ]:
from llama_index.core.base.response.schema import RESPONSE_TYPE,  Response
from src.kb_engine.patched_node_with_score import NodeWithScore

from pydantic import BaseModel
class ChunkInfo(BaseModel):
    file_path: str
    file_name: str
    page_nr: int

    def from_node(node: NodeWithScore):
        print(node.metadata)
        return ChunkInfo(
            file_path=node.metadata.get("filepath"),
            file_name=node.metadata.get("origin",{}).get("filename"),
            page_nr = node.metadata.get("doc_items", [])[0].get("prov", {})[0].get("page_no")
            
        )

    def get_page_image(self):
        file = Path(self.file_path)
        page_nr = self.page_nr
from devtools import debug
debug(result.source_nodes[0].metadata.get("doc_items", [])[0].get("prov", {})[0].get("page_no"))


/var/folders/bf/4pk3n3917lv30ykyw5g_g0mw0000gn/T/ipykernel_46809/1040368687.py:21 <module>
    result.source_nodes[0].metadata.get("doc_items", [])[0].get("prov", {})[0].get("page_no"): 4 (int)


4

In [57]:

for node in result.source_nodes:
    
    print("--------------------------------")

    debug(ChunkInfo.from_node(node))
    print("--------------------------------")



--------------------------------
{'filepath': 'data/uploads/Flux_Capacitor_Technical_Reference_Manual.pdf', 'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/12', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 4, 'bbox': {'l': 78.0, 't': 527.0699705078125, 'r': 553.0, 'b': 493.81997050781246, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 203]}]}], 'headings': ['3. Mechanical & Thermal Properties'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 4142608075159877010, 'filename': 'Flux_Capacitor_Technical_Reference_Manual.pdf'}}
/var/folders/bf/4pk3n3917lv30ykyw5g_g0mw0000gn/T/ipykernel_46809/197488660.py:5 <module>
    ChunkInfo.from_node(node): ChunkInfo(
        file_path='data/uploads/Flux_Capacitor_Technical_Reference_Manual.pdf',
        file_name='Flux_Capacitor_Technical_Reference_Manual.pdf',
        page_nr=4,
    ) (ChunkInfo)
----------------

In [ ]:
[
    {
        'self_ref': '#/texts/12',
        'parent': {
            '$ref': '#/body'
        },
        'children': [],
        'content_layer': 'body',
        'label': 'text',
        'prov': [
            {
                'page_no': 4,
                'bbox': {
                    'l': 78.0,
                    't': 527.0699705078125,
                    'r': 553.0,
                    'b': 493.81997050781246,
                    'coord_origin': 'BOTTOMLEFT'
                },
                'charspan': [
                    0,
                    203
                ]
            }
        ]
    }
]